# Modulo 8 — Sesion 6 · Ejercicio A

| | |
|---|---|
| **Tiempo** | 40 minutos |
| **Punto de partida** | Dashboard publicado de S5 con las graficas ya funcionando |
| **Producto esperado** | 2 graficas anotadas integradas en el dashboard publicado |

### Objetivo
Agregar anotaciones narrativas a la grafica de barras de rotacion
y al scatter de riesgo para que el Director de RH pueda entenderlas
sin explicacion adicional.

1. Mostrar primero la grafica SIN anotaciones — preguntar si el Director la entiende
2. Agregar las anotaciones y preguntar de nuevo — el contraste es el aprendizaje
3. Enfatizar: las anotaciones no son decoracion, son la historia de los datos
4. Punto critico: verificar que las anotaciones se regeneran al cambiar filtros

- annotation_position con valor invalido — usar exactamente: 'top right', 'bottom left'
- opacity demasiado alto en add_hrect — usar 0.15 a 0.25
- add_annotation con coordenadas incorrectas — verificar el valor exacto de x e y


## ▶ EJECUTA — Grafica 1: Barras con anotaciones

Primero la version sin anotaciones, luego con anotaciones.
Observa la diferencia — ¿el Director entiende sin explicacion?

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px

df = pd.read_csv('../hr_ibm.csv')

# Reutilizar calcular_score() de la practica C2

def calcular_score(df):
    d = df.copy()
    d['n_overtime'] = (d['OverTime'] == 'Yes').astype(int)*100
    d['n_jobsat'] = (4 - d['JobSatisfaction'])/3 *100
    d['n_years'] = ((3-d['YearsAtCompany'])/3*100).clip(0,100)
    d['n_wlb'] = (4 - d['WorkLifeBalance']) /3  *100
    d['n_envsat'] = (4 - d['EnvironmentSatisfaction'])/3 * 100

    d['score_riesgo'] = (
        d['n_overtime']*0.30 + d['n_jobsat']*0.25 +
        d['n_years']*0.20 + d['n_wlb']*0.15 +
        d['n_envsat']*0.10
    ).round(1)

    d['nivel_riesgo'] = pd.cut(d['score_riesgo'], bins=[0, 39, 69, 100], labels=['Bajo', 'Medio', 'Alto'], include_lowest=True)

    return d

df_score = calcular_score(df)

print(f"Dataset listo Total de Empleados: {len(df_score)}")
print(f"\n Distribución por nivel de riesgo")
print(df_score['nivel_riesgo'].value_counts().sort_index().to_string())
print(f"\n Score promedio global {df_score['score_riesgo'].mean():.1f}")
print(f"Salario mensual promedio: ${df_score['MonthlyIncome'].mean():,.0f}")





Dataset listo Total de Empleados: 1470

 Distribución por nivel de riesgo
nivel_riesgo
Bajo     998
Medio    430
Alto      42

 Score promedio global 32.3
Salario mensual promedio: $6,503


In [3]:
# PASO 1 — Grafica de barras SIN anotaciones (version original)
# Mostrar primero — preguntar al grupo si el Director la entiende
rot = (df.groupby('Department')['Attrition'].apply(lambda x: (x=='Yes').mean()*100).reset_index())

rot.columns = ['Department', 'Tasa_Rotacion']

rot = rot.sort_values('Tasa_Rotacion', ascending=False)

print('Tasa de rotación por departamento')
print(rot.to_string(index=False))
print(f"\n Departmaento critico: {rot.iloc[0]['Department']}   ({rot.iloc[0]['Tasa_Rotacion']:.1f}%)")

fig_sin = px.bar(rot, x='Department', y='Tasa_Rotacion', 
                 title='Tasa de Rotación por departamento (%)',
                 color='Department', text_auto='.1f',
                 labels={'Tasa_Rotacion' : 'Tasa (%) ', 'Department' : 'Deparmento'})
fig_sin.update_layout(height=420, showlegend=False)
fig_sin.show()





Tasa de rotación por departamento
            Department  Tasa_Rotacion
                 Sales      20.627803
       Human Resources      19.047619
Research & Development      13.839750

 Departmaento critico: Sales   (20.6%)


In [6]:
# PASO 2 — Grafica de barras CON anotaciones (version storytelling)


rot = (df.groupby('Department')['Attrition'].apply(lambda x: (x=='Yes').mean()*100).reset_index())

rot.columns = ['Department', 'Tasa_Rotacion']

rot = rot.sort_values('Tasa_Rotacion', ascending=False)

depto_critico = rot.iloc[0]['Department']
tasa_critica = rot.iloc[0]['Tasa_Rotacion']

print('Tasa de rotación por departamento')
print(rot.to_string(index=False))
print(f"\n Departmaento critico: {rot.iloc[0]['Department']}   ({rot.iloc[0]['Tasa_Rotacion']:.1f}%)")

fig_con = px.bar(rot, x='Department', y='Tasa_Rotacion', 
                 title='Tasa de Rotación por departamento (%)',
                 color='Department', text_auto='.1f',
                 labels={'Tasa_Rotacion' : 'Tasa (%) ', 'Department' : 'Deparmento'})
fig_con.update_layout(height=420, showlegend=False)
fig_con.show()


# Anotacion 1: linea de referencia del sector

fig_con.add_hline(y=10, line_dash='dot', line_color='#F57F18', line_width=2,
              annotation_text='Meta sector: 10%',
              annotation_position='top right',
              annotation_font_color='#F57F18', annotation_font_size=12)


# Anotacion 2: etiqueta sobre la barra critica

fig_con.add_annotation(
    x=depto_critico, y=tasa_critica,
    text=f'{depto_critico} : {tasa_critica:.1f}%   --- {tasa_critica-10:.1f} puntos sobre la meta',
    showarrow=True, arrowhead=2, arrowcolor='#880000',
    bgcolor='#FFEBBE', font=dict(color='#880000', size=12), yshift=10
)
fig_con.update_layout(height=420, showlegend=False)
fig_con.show()


Tasa de rotación por departamento
            Department  Tasa_Rotacion
                 Sales      20.627803
       Human Resources      19.047619
Research & Development      13.839750

 Departmaento critico: Sales   (20.6%)


### ✏️ TU TURNO — Integra en tu app.py

In [ ]:
# PASO 3 — Version para integrar en el server() de la app Shiny



## ▶ EJECUTA — Grafica 2: Scatter con cuadrantes

In [7]:
# PASO 4 — Scatter SIN anotaciones (version original)
print("Resumen de datos para el Scatter")
print(f"Empleados en Riesgo Alto : {(df_score['nivel_riesgo']=='Alto').sum()}")
print(f"Empleados en Riesgo Medio : {(df_score['nivel_riesgo']=='Medio').sum()}")
print(f"Empleados en Riesgo Bajo : {(df_score['nivel_riesgo']=='Bajo').sum()}")

fig_scatter_sin = px.scatter(df_score,
                             x='MonthlyIncome', y='score_riesgo', color='nivel_riesgo',
                             color_discrete_map={'Alto' : '#C62828', 'Medio' : '#F57F17', 'Bajo': '#2E7D32'},
                             title='Score de riesgo vs Salario Mensual',
                              labels={'MonthlyIncome' : 'Salario (USD)', 'score_riesgo' : 'Score de Riesgo' },
                                opacity=0.6)
fig_scatter_sin.update_layout(height=420)
fig_scatter_sin.show()


Resumen de datos para el Scatter
Empleados en Riesgo Alto : 42
Empleados en Riesgo Medio : 430
Empleados en Riesgo Bajo : 998


In [14]:
# PASO 5 — Scatter CON anotaciones — zona de riesgo y cuadrantes
sal_prom = df_score['MonthlyIncome'].mean()


fig_scatter_con = px.scatter(df_score,
                             x='MonthlyIncome', y='score_riesgo', color='nivel_riesgo',
                             color_discrete_map={'Alto' : '#C62828', 'Medio' : '#F57F17', 'Bajo': '#2E7D32'},
                             title='Score de riesgo vs Salario Mensual',
                              labels={'MonthlyIncome' : 'Salario (USD)', 'score_riesgo' : 'Score de Riesgo' },
                                opacity=0.6)
fig_scatter_con.update_layout(height=420)
fig_scatter_con.show()
# Zona de riesgo alto
fig_scatter_con.add_hrect(y0=70, y1=100, fillcolor='#FFBEEE', opacity=0.4,
                          line_width=0, annotation_text='ZONA DE RIESGO ALTO',
                          annotation_position='top right', annotation_font_color='#880000')


# Linea del salario promedio
fig_scatter_con.add_vline(x=sal_prom, line_dash='dash', line_color='#1A5F8A',
                          annotation_text=f'Salario promedio: ${sal_prom:,.0f}',
                          annotation_position='top right', annotation_font_color='#1A5F8A')


# Etiqueta del cuadrante de maxima prioridad
fig_scatter_con.add_annotation(x=sal_prom*1.5, y=85,
                               text='MAX PRIORIDAD', showarrow=False,
                               font=dict(color='#880000', size=11),
                               bgcolor='rgba(255, 255, 238, 0.9)')
fig_scatter_con.update_layout(height=420)
fig_scatter_con.show()



### ✏️ TU TURNO — Integra en tu app.py

In [ ]:
# PASO 6 — Version para integrar en el server() de la app Shiny



### Autoevaluacion
| Criterio | Cumple |
|----------|--------|
| Grafica de barras tiene linea de referencia en 10% | ☐ |
| Grafica de barras tiene anotacion sobre la barra critica | ☐ |
| Scatter tiene zona sombreada de riesgo alto (score>70) | ☐ |
| Scatter tiene linea vertical del salario promedio | ☐ |
| Ambas funciones estan integradas en app.py | ☐ |
| Las anotaciones se regeneran al cambiar filtros | ☐ |